In [2]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# 토크나이제이션

legal_sentence = '법원은 피고인에게 징역 3년과 벌금 500만원을 선고했다'
tokens = legal_sentence.split()
reg_tokens = re.findall(r'[가-힣A-Za-z0-9]+', legal_sentence)
print(f'원문 : {legal_sentence}')
print('\n공백기준')
print(tokens)
print(f'개수 : {len(tokens)}')
print('\n정규표현식 토큰')
print(reg_tokens)
print(f'개수 : {len(reg_tokens)}')

legal_keyword = ['법원','피고인','징역','벌금','선고']
keyword_presence = { keyword : keyword in legal_sentence for keyword in legal_keyword}
print('\n법률 핵심어 포함 여부:')
print(keyword_presence)


원문 : 법원은 피고인에게 징역 3년과 벌금 500만원을 선고했다

공백기준
['법원은', '피고인에게', '징역', '3년과', '벌금', '500만원을', '선고했다']
개수 : 7

정규표현식 토큰
['법원은', '피고인에게', '징역', '3년과', '벌금', '500만원을', '선고했다']
개수 : 7

법률 핵심어 포함 여부:
{'법원': True, '피고인': True, '징역': True, '벌금': True, '선고': True}


In [ ]:
# 간이 임베딩 과 의미유사도 : 임베딩(문자를 숫자로 바꿈) 과정을 확인
# BOW(Bag of Words) 표현으로 실습

embedding_examples = pd.DataFrame([
    {'id': 'S1', 'label': 'PROC', 'text': '위반한 자는 3년 이하의 징역 또는 벌금에 처한다.'},
    {'id': 'S2', 'label': 'PROC', 'text': '거짓 신고를 한 자는 1천만원 이하의 벌금에 처한다.'},
    {'id': 'S3', 'label': 'ORG', 'text': '분쟁 조정을 위하여 조정위원회를 설치한다.'},
    {'id': 'S4', 'label': 'ORG', 'text': '위원회는 위원장 1명을 포함하여 15명 이내의 위원으로 구성한다.'},
    {'id': 'S5', 'label': 'RIGHT', 'text': '근로자는 안전한 환경에서 일할 권리를 가진다.'},
])
# 간이임베딩
vocabulary = ['징역', '벌금', '처한다', '위원회', '설치', '구성', '권리', '근로자', '안전']

for word in vocabulary:
    embedding_examples[word] = embedding_examples['text'].str.contains(word).astype(int)

vector_columns = vocabulary
print('문장별 간이 임베딩 벡터')
print(embedding_examples[['id','label','text'] + vector_columns])

def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a, dtype=float)
    b = np.array(vec_b, dtype=float)
    denomiator = np.linalg.norm(a) * np.linalg.norm(b)
    if denomiator == 0:
        return 0.0
    return float(np.dot(a,b) / denomiator)

similarity_rows = []
for i in range(len(embedding_examples)):
    row_i = embedding_examples.iloc[i]
    for j in range(i+1, len(embedding_examples)):        
        row_j = embedding_examples.iloc[j]
        sim = cosine_similarity(row_i[vector_columns],row_j[vector_columns])
        similarity_rows.append({
            'pair' : f'{row_i["id"]}-{row_j["id"]}',
            'labels' : f'{row_i["label"]}-{row_j["label"]}',
            'simularity' : round(sim,3),
            'text_a': row_i['text'],
            'text_b': row_j['text']
        })

similarity_df = pd.DataFrame(similarity_rows).sort_values('simularity',ascending=False)
similarity_df


문장별 간이 임베딩 벡터
   id  label                                  text  징역  벌금  처한다  위원회  설치  구성  \
0  S1   PROC          위반한 자는 3년 이하의 징역 또는 벌금에 처한다.   1   1    1    0   0   0   
1  S2   PROC         거짓 신고를 한 자는 1천만원 이하의 벌금에 처한다.   0   1    1    0   0   0   
2  S3    ORG               분쟁 조정을 위하여 조정위원회를 설치한다.   0   0    0    1   1   0   
3  S4    ORG  위원회는 위원장 1명을 포함하여 15명 이내의 위원으로 구성한다.   0   0    0    1   0   1   
4  S5  RIGHT             근로자는 안전한 환경에서 일할 권리를 가진다.   0   0    0    0   0   0   

   권리  근로자  안전  
0   0    0   0  
1   0    0   0  
2   0    0   0  
3   0    0   0  
4   1    1   1  


,pair,labels,simularity,text_a,text_b
0,S1-S2,PROC-PROC,0.816,위반한 자는 3년 이하의 징역 또는 벌금에 처한다.,거짓 신고를 한 자는 1천만원 이하의 벌금에 처한다.
7,S3-S4,ORG-ORG,0.500,분쟁 조정을 위하여 조정위원회를 설치한다.,위원회는 위원장 1명을 포함하여 15명 이내의 위원으로 구성한다.
1,S1-S3,PROC-ORG,0.000,위반한 자는 3년 이하의 징역 또는 벌금에 처한다.,분쟁 조정을 위하여 조정위원회를 설치한다.
2,S1-S4,PROC-ORG,0.000,위반한 자는 3년 이하의 징역 또는 벌금에 처한다.,위원회는 위원장 1명을 포함하여 15명 이내의 위원으로 구성한다.
4,S2-S3,PROC-ORG,0.000,거짓 신고를 한 자는 1천만원 이하의 벌금에 처한다.,분쟁 조정을 위하여 조정위원회를 설치한다.
3,S1-S5,PROC-RIGHT,0.000,위반한 자는 3년 이하의 징역 또는 벌금에 처한다.,근로자는 안전한 환경에서 일할 권리를 가진다.
5,S2-S4,PROC-ORG,0.000,거짓 신고를 한 자는 1천만원 이하의 벌금에 처한다.,위원회는 위원장 1명을 포함하여 15명 이내의 위원으로 구성한다.
6,S2-S5,PROC-RIGHT,0.000,거짓 신고를 한 자는 1천만원 이하의 벌금에 처한다.,근로자는 안전한 환경에서 일할 권리를 가진다.
8,S3-S5,ORG-RIGHT,0.000,분쟁 조정을 위하여 조정위원회를 설치한다.,근로자는 안전한 환경에서 일할 권리를 가진다.
9,S4-S5,ORG-RIGHT,0.000,위원회는 위원장 1명을 포함하여 15명 이내의 위원으로 구성한다.,근로자는 안전한 환경에서 일할 권리를 가진다.


In [ ]:
# LLM 작동 원리 - 자동회귀식 다음 토큰 예측
# 앞의 토큰을 보고 다음 토큰을 예측한다는 규칙모델을 실습
# 실제로 llm 거대한 신경망으로 다음 토큰을 예측 
# 우리는 법률문장 몇개에서 bigram빈도를 세어서 특정토큰뒤에 어떤 토큰이 자주나오는지 확인

autoregressive_corpus = [
    '이 법은 국민의 권리를 보호함을 목적으로 한다',
    '이 법은 개인정보를 안전하게 보호함을 목적으로 한다',
    '위원회는 분쟁 조정을 위하여 설치한다',
    '위원회는 위원장 1명을 포함하여 구성한다',
    '위반한 자는 벌금에 처한다',
    '위반한 자는 징역에 처한다',
]
def tokenize_for_bigram(text):
    return re.findall(f'[가-힣A-Za-z09]+',text)

# 현재 토큰이 주어졌을때 다음 토큰이 무엇인지 빈도수 계산
bigram_counts = {}
for sentence in autoregressive_corpus:
    tokens = ['<START>'] + tokenize_for_bigram(sentence) + ['<END>']
    for current_token, next_token in zip(tokens, tokens[1:]):  # <START>, '이'   '이','법은'
        bigram_counts.setdefault(current_token,Counter())[next_token] += 1
print('특정 토큰 뒤에 올 가능성인 높은 후보')
for token in ['<START>', '이','법은','위원회는','위반한','자는' ]:
    print(f'{token} -> {bigram_counts.get(token,Counter()).most_common()}')

{'<START>': Counter({'이': 2, '위원회는': 2, '위반한': 2}),
 '이': Counter({'법은': 2}),
 '법은': Counter({'국민의': 1, '개인정보를': 1}),
 '국민의': Counter({'권리를': 1}),
 '권리를': Counter({'보호함을': 1}),
 '보호함을': Counter({'목적으로': 2}),
 '목적으로': Counter({'한다': 2}),
 '한다': Counter({'<END>': 2}),
 '개인정보를': Counter({'안전하게': 1}),
 '안전하게': Counter({'보호함을': 1}),
 '위원회는': Counter({'분쟁': 1, '위원장': 1}),
 '분쟁': Counter({'조정을': 1}),
 '조정을': Counter({'위하여': 1}),
 '위하여': Counter({'설치한다': 1}),
 '설치한다': Counter({'<END>': 1}),
 '위원장': Counter({'명을': 1}),
 '명을': Counter({'포함하여': 1}),
 '포함하여': Counter({'구성한다': 1}),
 '구성한다': Counter({'<END>': 1}),
 '위반한': Counter({'자는': 2}),
 '자는': Counter({'벌금에': 1, '징역에': 1}),
 '벌금에': Counter({'처한다': 1}),
 '처한다': Counter({'<END>': 2}),
 '징역에': Counter({'처한다': 1})}